Оцените эксперимент «Sending email (correct link)» с использованием CUPED. В качестве ковариаты используйте выручку пользователей за 4 недели до эксперимента.

In [47]:
import os
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
from scipy import stats

URL_BASE = 'https://raw.githubusercontent.com/ab-courses/simulator-ab-datasets/main/2022-05-03/'

def read_database(file_name):
    return pd.read_csv(os.path.join(URL_BASE, file_name))

df_sales = read_database('2022-05-03T12_df_sales.csv')
df_sales['date'] = pd.to_datetime(df_sales['date'])
df_users = read_database('experiment_users.csv')

begin_date = datetime(2022, 4, 25)
end_date = datetime(2022, 5, 2)
begin_cov_date = begin_date - timedelta(days=28)

df_metrics = (
    df_sales
    [(df_sales['date'] >= begin_date) & (df_sales['date'] < end_date)]
    .groupby('user_id')[['price']].sum()
    .rename(columns={'price': 'metric'})
    .reset_index()
)
df_cov = (
    df_sales
    [(df_sales['date'] >= begin_cov_date) & (df_sales['date'] < begin_date)]
    .groupby('user_id')[['price']].sum()
    .rename(columns={'price': 'cov'})
    .reset_index()
)

df = pd.merge(df_users, df_metrics, on='user_id', how='left')
df = pd.merge(df, df_cov, on='user_id', how='left').fillna(0)


def calculate_theta(y_control, y_pilot, x_control, x_pilot):
    """Вычисляем Theta по данным двух групп.

    y_control - значения метрики во время пилота на контрольной группе
    y_pilot - значения метрики во время пилота на пилотной группе
    x_control - значения ковариант на контрольной группе
    x_pilot - значения ковариант на пилотной группе
    """
    y = np.hstack([y_control, y_pilot])
    x = np.hstack([x_control, x_pilot])
    covariance = np.cov(x, y)[0, 1]
    variance = x.var()
    theta = covariance / variance
    return theta


def check_cuped_test(df_control, df_pilot, covariate_column):
    """Проверяет гипотезу о равенстве средних с использованием CUPED.

    covariate_column - название стобца с ковариантой

    return - pvalue.
    """
    theta = calculate_theta(
        df_control['metric'], df_pilot['metric'],
        df_control[covariate_column], df_pilot[covariate_column]
    )
    metric_cuped_control = df_control['metric'] - theta * df_control[covariate_column]
    metric_cuped_pilot = df_pilot['metric'] - theta * df_pilot[covariate_column]
    _, pvalue = stats.ttest_ind(metric_cuped_control, metric_cuped_pilot)
    return pvalue


pvalue = check_cuped_test(df[df['pilot'] == 0], df[df['pilot'] == 1], 'cov')
print(f'pvalue = {pvalue:0.4f}')


pvalue = 0.0539


In [51]:
print(df[df['pilot'] == 0].shape)

(54736, 4)


In [52]:
df[df['pilot'] == 1].shape

(54631, 4)